# Python for Data Engineering -- Practice Exercises (Questions)
Student: Newana Tandukar
Day: 7

Task prompts drawn from the Day 7 class notebook (NorthStar Outfitters
scenario), one exercise per section. Fill in the empty code cell(s)
under each question. Solutions are in
`python-for-data-engineering-solved.ipynb`.

Module 2 needs the local mock partner API running
(`python scripts/mock_api_server.py`, from the original class setup) and
Module 3 needs a local PostgreSQL instance reachable at `localhost:5432`
(see this folder's `README.md`).

# Python for Data Engineering

### A hands-on class for the data team at **NorthStar Outfitters** (fictional online outdoor-gear retailer)

**Duration:** ~2 hours
**You should already be comfortable with:** Python data structures (lists, dicts, tuples) and basic OOP (classes, methods).
**You do NOT need any prior pandas / SQL / API experience** -- that's what today is for.

---

## The scenario

You've just joined the data engineering team. The analytics and finance teams need clean, trustworthy data to do their jobs, but right now that data is scattered:

- Order and customer records live in **CSV, Excel, and JSON files** exported from different systems.
- Two partner services -- a currency-exchange provider and a shipping carrier -- only expose their data through an **HTTP API**.
- The warehouse's shipment system writes **raw text log lines**, not tidy tables.
- Everything eventually needs to land in a **PostgreSQL database** so the BI team can query it with SQL and BI tools.

Today you'll build a small end-to-end pipeline that touches every one of those data sources. This is, in miniature, what a data engineer does every day.

## Agenda (~2 hours)

| Time | Module | What you'll do |
|---|---|---|
| 0:00 - 0:05 | Setup check | Confirm your environment is ready |
| 0:05 - 1:00 | **Module 1** -- Working with Structured Data | pandas, reading CSV/Excel/JSON, filtering, grouping, missing data |
| 1:00 - 1:40 | **Module 2** -- APIs & Semi-Structured Data | `requests`, parsing JSON, regex on log files |
| 1:40 - 2:05 | **Module 3** -- Connecting Python to Databases | SQLAlchemy/psycopg2, loading data, safe parameterized SQL |
| 2:05 - 2:10 | Wrap-up | Recap + extension ideas |

## Before you start

Make sure you've completed the **one-time setup** in `README.md` in this folder:

1. `pip install -r requirements.txt`
2. `docker compose up -d` (starts PostgreSQL for Module 3)
3. `python scripts/mock_api_server.py` running in its own terminal (powers Module 2)

If you haven't done those yet, pause and do them now -- the setup-check cell below will tell you what's missing.

In [ ]:
# --- Setup check -------------------------------------------------------
# Run this first. It doesn't teach anything new -- it just confirms your
# environment is ready so you're not debugging setup issues mid-class.

import importlib
import os

import pandas as pd
import numpy as np

REQUIRED_PACKAGES = ["pandas", "numpy", "openpyxl", "requests", "sqlalchemy", "psycopg2"]
missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]

print(f"pandas version:  {pd.__version__}")
print(f"numpy version:   {np.__version__}")

if missing:
    print(f"\n[MISSING PACKAGES] {missing} -- run: pip install -r requirements.txt")
else:
    print("\nAll required packages are installed.")

DATA_DIR = os.path.join("data")
expected_files = ["orders.csv", "customers.xlsx", "products.json", "shipment_logs.txt"]
present = os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else []
missing_files = [f for f in expected_files if f not in present]

if missing_files:
    print(f"[MISSING DATA FILES] {missing_files}")
    print("  -> run: python scripts/generate_sample_data.py")
else:
    print("All data files found in data/.")

# Module 1: Working with Structured Data

**Scenario:** Your first task is to get comfortable moving NorthStar's core business data -- orders, customers, and products -- into a shape the analytics team can actually use. This data currently lives in three different file formats, which is extremely common: different upstream systems export in whatever format is convenient for *them*, not for you.

## 1.1 Introduction to pandas: `Series` and `DataFrame`

Two objects do almost all the work in pandas:

- A **`Series`** is a single labeled column of data -- think of it as a list, but every value has an index label attached.
- A **`DataFrame`** is a table: a collection of `Series` that all share the same index, like a spreadsheet or a SQL table.

Let's see this with something small and concrete before touching real files: NorthStar's website visits for one week.

**1.** NorthStar's website visits for one week are `[812, 790, 905, 1140, 1225, 1560, 980]` for Mon-Sun. Build a `pd.Series` of these visits (indexed Mon-Sun, named `site_visits`) and print the busiest day with `idxmax()`/`max()`. Then build a `pd.DataFrame` with three columns -- `site_visits`, `orders_placed` (`[34, 29, 41, 55, 63, 88, 47]`), and `marketing_spend_usd` (`[150, 150, 150, 300, 300, 500, 200]`) -- and add a computed `conversion_rate_pct` column (`orders_placed / site_visits * 100`, rounded to 2 decimals).

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**2.** Read `data/orders.csv` into a DataFrame with `pd.read_csv()`, parsing `order_date` as a real date (`parse_dates=["order_date"]`). Print its shape and inspect it with `.head()` and `.info()`.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**3.** Read `data/customers.xlsx` (sheet `"customers"`) into a DataFrame with `pd.read_excel()`. Print its shape and `.head()`.

In [ ]:
# TODO: your code here

**4.** Load `data/products.json` with `json.load()` -- notice each product has a nested `"supplier"` object. Flatten it into a tidy DataFrame with `pd.json_normalize()`.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**5.** Pull just the `order_id`, `country`, and `order_status` columns from `orders_df`.

In [ ]:
# TODO: your code here

**6.** Show every order from Germany.

In [ ]:
# TODO: your code here

**7.** Now just the Delivered orders from the USA -- combine both conditions with `&`, wrapping each condition in parentheses.

In [ ]:
# TODO: your code here

**8.** Add a `gross_amount` column with the order's gross value before discount (`quantity * unit_price`), as a vectorized operation (no `for` loop).

In [ ]:
# TODO: your code here

**9.** Compute total `gross_amount` **per country**, sorted highest first, using `groupby()`.

In [ ]:
# TODO: your code here

**10.** Group by `country` **and** `order_status` at once, and compute `order_count`, `total_revenue`, and `avg_order_value` together with `.agg()`.

In [ ]:
# TODO: your code here

**11.** Which loyalty tier generates the most revenue? Merge `orders_df` with `customers_df` (bringing in `loyalty_tier`) using `pd.merge()` (a left join on `customer_id`), then group by `loyalty_tier`.

In [ ]:
# TODO: your code here

**12.** Do the same idea joining in the product catalog: merge in `category`/`name` from `products_df` and find total revenue by category.

In [ ]:
# TODO: your code here

**13.** Find out exactly which columns in `orders_df` have missing values with `isna().sum()`. Then, on a copy (`orders_clean`), apply a different strategy per column: `discount_pct` missing means "no discount" (`fillna(0)`), `shipping_cost` should be filled with the median (robust to skew), and rows missing `customer_id` can't be attributed to anyone and should be dropped (`dropna`). Print how many rows were dropped and confirm no missing values remain.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**14.** Now that `discount_pct` has no missing values, safely compute a `total_amount` column: `quantity * unit_price * (1 - discount_pct)`, rounded to 2 decimals.

In [ ]:
# TODO: your code here

# Module 2: APIs and Semi-Structured Data

Not all of NorthStar's data lives in files: a currency-exchange
provider and a shipping-carrier tracker only expose their data through
an HTTP API, and the warehouse's shipment system writes raw text logs.

**Before running these cells:** start the local mock partner API with
`python scripts/mock_api_server.py` in its own terminal.

Confirm the mock API is reachable before relying on it for the rest of this module.

In [ ]:
# TODO: your code here

**15.** Finance wants revenue reported in each customer's local currency. Call `GET {API_BASE}/api/v1/exchange-rates?base=USD` with `requests`, print the URL and status code, and call `.raise_for_status()` so a failed call fails loudly.

In [ ]:
# TODO: your code here

**16.** Parse the exchange-rate response with `.json()`. It's a dict with a `"rates"` sub-dict -- use it, plus a `country -> currency` mapping, to convert each country's USD revenue (from Exercise 9) into its local currency.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**17.** The warehouse writes raw log lines to `data/shipment_logs.txt`, not a CSV. Read it into a list of lines and print the first 3.

In [ ]:
# TODO: your code here

**18.** Write a regex with named groups (`(?P<name>...)`) that captures `timestamp`, `level`, `order_id`, `customer_id`, `carrier`, `tracking_number`, `status`, `origin`, `dest`, `weight_kg`, and an optional `reason`. Apply it to every log line, and turn the matches into a DataFrame (`weight_kg` as `float`).

In [ ]:
# TODO: your code here

**19.** Using the `shipments_df` from the previous exercise: which shipments were delayed, and why?

In [ ]:
# TODO: your code here

**20.** Take one delayed shipment's `tracking_number` and call the carrier's tracking API for its full history. The response has a nested `"events"` list -- flatten it into rows with `pd.json_normalize(..., record_path="events", meta=[...])`, carrying `tracking_number`, `carrier`, and `estimated_delivery` onto every row.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

### Optional / bonus: calling a *real* public API

If you have internet access, call a real, free, no-API-key currency API
([frankfurter.app](https://www.frankfurter.app/)) with the same
`requests` pattern used above, to see that nothing about the technique
changes when the URL is real.

In [ ]:
# TODO: your code here

# Module 3: Connecting Python to Databases

Clean data sitting only in a notebook isn't useful to the BI team, who
query PostgreSQL directly. The last step is loading `orders_clean` into
a real Postgres database, and doing it safely.

**Before running these cells:** start Postgres with `docker compose up
-d` and confirm it's healthy with `docker compose ps`.

**21.** Build a SQLAlchemy `engine` from a `postgresql+psycopg2://` connection string (host `localhost`, port `5432`, database `northface_outfitters`), then run `SELECT version();` to confirm the connection works.

In [ ]:
# TODO: your code here

**22.** Create a `carriers` reference table if it doesn't exist, insert a handful of carrier rows (`ON CONFLICT DO NOTHING` so re-running is safe), then read it back with `pd.read_sql()`.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**23.** Write `orders_clean` into Postgres as a new `orders` table with `.to_sql(..., if_exists="replace")`, then confirm the row count landed correctly with a `SELECT COUNT(*)`.

In [ ]:
# TODO: your code here

**24.** Write a SQL query that groups `orders` by `country` and computes order count and total revenue, then load the result straight into a DataFrame with `pd.read_sql()`.

In [ ]:
# TODO: your code here

**25.** Write a `get_orders_for_customer_UNSAFE(customer_id)` helper that builds its SQL by f-string-concatenating `customer_id` directly into the query. Show it working for a normal ID, then pass it `"CUST-0001' OR '1'='1"` and show it returns every row in the table instead of one customer's.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**26.** Write a `get_orders_for_customer_SAFE(customer_id)` helper using `sqlalchemy.text()` with a named `:cid` placeholder and `params={'cid': customer_id}` instead of string formatting. Confirm the same malicious input now correctly returns zero rows.

In [ ]:
# TODO: your code here

**27.** Do the same thing directly with `psycopg2`: connect, then run a query using `%s` placeholders and pass the values as a separate tuple -- never with Python's `%` or f-string formatting.

In [ ]:
# TODO: your code here